# EA6D - Setup & Run

Setting up and running the EA6D repo on Google Colab or Kaggle.

**Before running:** Enable GPU — Colab: `Runtime > Change runtime type > T4 GPU` | Kaggle: `Settings > Accelerator > GPU T4 x2`

In [ ]:
!nvidia-smi

## 1. Clone repo & install dependencies

In [ ]:
!git clone https://github.com/Sweetdevil144/EA6D.git
%cd EA6D

In [ ]:
!pip install -q accelerate einops ema-pytorch fvcore scipy albumentations \
    scikit-learn tqdm transforms3d open3d pytorch-lightning \
    torchmetrics lpips pyyaml tensorboard huggingface_hub

## 2. Download & preprocess Linemod dataset

EA6D uses the **Linemod (LM)** dataset from the [BOP Challenge](https://bop.felk.cvut.cz/datasets/).
BOP provides it in their own format, so we convert to what EA6D expects:

```
can/
  image/   # RGB input images (condition for the diffusion model)
  clean/   # object isolated from background via mask (diffusion target)
  RT/      # 4x4 pose matrices as .txt files
```

Linemod object IDs: 1=ape, 2=benchvise, 4=camera, **5=can**, 6=cat, 8=driller, 9=duck, 10=eggbox, 11=glue, 12=holepuncher, 13=iron, 14=lamp, 15=phone

In [ ]:
import os

BASE = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
BOP_DIR = f"{BASE}/bop_lm"
os.makedirs(BOP_DIR, exist_ok=True)

# Use pre-uploaded Kaggle dataset if available, otherwise download
KAGGLE_DATASET = "/kaggle/input/datasets/abhinavpandey144/ea6d-dataset"
if os.path.exists(KAGGLE_DATASET):
    BOP_DIR = KAGGLE_DATASET
    print(f"Using uploaded Kaggle dataset at {BOP_DIR}")
else:
    from huggingface_hub import hf_hub_download
    for fname in ["lm_base.zip", "lm_test_bop19.zip"]:
        print(f"Downloading {fname}...")
        path = hf_hub_download(
            repo_id="bop-benchmark/lm",
            filename=fname,
            repo_type="dataset",
            local_dir=f"{BASE}/bop_downloads",
        )
        print(f"  -> {path} ({os.path.getsize(path) / 1e6:.1f} MB)")
        !unzip -q -o {path} -d {BOP_DIR}

!find {BOP_DIR} -name 'scene_gt.json' | head -5

In [ ]:
import json, os
import numpy as np
from PIL import Image
from pathlib import Path

def convert_bop_to_ea6d(bop_scene_dir, output_dir, obj_id):
    """
    Convert BOP scene to EA6D folder structure.
    """
    bop_scene_dir = Path(bop_scene_dir)
    output_dir = Path(output_dir)

    (output_dir / 'image').mkdir(parents=True, exist_ok=True)
    (output_dir / 'clean').mkdir(parents=True, exist_ok=True)
    (output_dir / 'RT').mkdir(parents=True, exist_ok=True)

    gt_path = bop_scene_dir / 'scene_gt.json'
    assert gt_path.exists(), f"scene_gt.json not found at {gt_path}"

    with open(gt_path) as f:
        scene_gt = json.load(f)

    count = 0
    for img_id_str, gt_list in scene_gt.items():
        img_id = int(img_id_str)

        for gt_idx, gt in enumerate(gt_list):
            if gt['obj_id'] != obj_id:
                continue

            rgb_path = bop_scene_dir / 'rgb' / f'{img_id:06d}.png'
            mask_path = bop_scene_dir / 'mask_visib' / f'{img_id:06d}_{gt_idx:06d}.png'

            if not rgb_path.exists():
                continue

            rgb = Image.open(rgb_path).convert('RGB')
            rgb_np = np.array(rgb)

            if mask_path.exists():
                mask = np.array(Image.open(mask_path).convert('L'))
                clean_np = rgb_np.copy()
                clean_np[mask <= 127] = 255
            else:
                clean_np = rgb_np.copy()

            R = np.array(gt['cam_R_m2c']).reshape(3, 3)
            t = np.array(gt['cam_t_m2c']).reshape(3, 1)
            pose = np.eye(4)
            pose[:3, :3] = R
            pose[:3, 3] = t.flatten()

            fname = f'{count:06d}'
            rgb.save(output_dir / 'image' / f'{fname}.png')
            Image.fromarray(clean_np).save(output_dir / 'clean' / f'{fname}.png')
            np.savetxt(output_dir / 'RT' / f'{fname}.txt', pose)
            count += 1

    return count


OBJECT_ID = 5
OBJECT_NAME = 'can'

BASE = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
BOP_DIR = f"{BASE}/bop_lm"

import glob as g
scene_gts = g.glob(f"{BOP_DIR}/**/scene_gt.json", recursive=True)
print(f"Found {len(scene_gts)} scenes: {scene_gts[:5]}")

BOP_SCENE = None
for sg in scene_gts:
    with open(sg) as f:
        data = json.load(f)
    obj_ids = set()
    for gt_list in data.values():
        for gt in gt_list:
            obj_ids.add(gt['obj_id'])
    if OBJECT_ID in obj_ids:
        BOP_SCENE = str(Path(sg).parent)
        print(f"Object {OBJECT_ID} found in: {BOP_SCENE}")
        break

assert BOP_SCENE is not None, f"Object {OBJECT_ID} not found in any scene"

DATA_DIR = f"{BASE}/ea6d_data/{OBJECT_NAME}"
TEST_DIR = f"{BASE}/ea6d_data/{OBJECT_NAME}_test"
OUTPUT_DIR = f"{BASE}/ea6d_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nConverting object '{OBJECT_NAME}' (id={OBJECT_ID})...")
total = convert_bop_to_ea6d(BOP_SCENE, DATA_DIR, OBJECT_ID)
print(f"{total} images converted")

import shutil
n_test = max(1, total // 5)
n_train = total - n_test

os.makedirs(f"{TEST_DIR}/image", exist_ok=True)
os.makedirs(f"{TEST_DIR}/RT", exist_ok=True)

for i in range(n_train, total):
    fname = f"{i:06d}"
    shutil.move(f"{DATA_DIR}/image/{fname}.png", f"{TEST_DIR}/image/{fname}.png")
    shutil.move(f"{DATA_DIR}/RT/{fname}.txt", f"{TEST_DIR}/RT/{fname}.txt")
    clean_path = f"{DATA_DIR}/clean/{fname}.png"
    if os.path.exists(clean_path):
        os.remove(clean_path)

print(f"Train: {n_train} | Test: {n_test}")
for d in [DATA_DIR, TEST_DIR]:
    for sub in ['image', 'clean', 'RT']:
        p = os.path.join(d, sub)
        n = len(os.listdir(p)) if os.path.exists(p) else 0
        print(f"  {p}: {n} files")

## 3. Training & Inference

Three stages run sequentially: VAE → Diffusion → Inference. Safe to **Run All**.

In [ ]:
import yaml, os

BASE = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(f'{BASE}/EA6D')

DATA_DIR = f"{BASE}/ea6d_data/can"
OUTPUT_DIR = f"{BASE}/ea6d_output"
os.makedirs(f"{OUTPUT_DIR}/step1", exist_ok=True)

vae_train_cfg = {
    'model': {
        'embed_dim': 3,
        'lossconfig': {
            'disc_start': 50001,
            'kl_weight': 0.000001,
            'disc_weight': 0.5,
            'disc_in_channels': 3,
        },
        'ddconfig': {
            'double_z': True,
            'z_channels': 3,
            'resolution': [256, 256],
            'in_channels': 3,
            'out_ch': 3,
            'ch': 128,
            'ch_mult': [1, 2, 4],
            'num_res_blocks': 2,
            'attn_resolutions': [],
            'dropout': 0.0,
        },
        'ckpt_path': None,
    },
    'data': {
        'name': 'edge',
        'img_folder': DATA_DIR,
        'augment_horizontal_flip': True,
        'batch_size': 1,
    },
    'trainer': {
        'gradient_accumulate_every': 2,
        'lr': 5e-6,
        'min_lr': 5e-7,
        'train_num_steps': 10000,
        'save_and_sample_every': 2000,
        'log_freq': 500,
        'results_folder': f'{OUTPUT_DIR}/step1/',
        'amp': False,
        'fp16': False,
    },
}
with open('configs/colab_vae_train.yaml', 'w') as f:
    yaml.dump(vae_train_cfg, f, default_flow_style=False)
print("VAE config written.")

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
BASE = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(f'{BASE}/EA6D')
!python train_vae.py --cfg configs/colab_vae_train.yaml

## 4. Step 2 — Train the Diffusion Model

Conditional UNet trained in latent space using the frozen VAE from Step 1.
The cell below automatically picks up the latest VAE checkpoint.

In [ ]:
import glob, yaml, os

BASE = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(f'{BASE}/EA6D')

DATA_DIR = f"{BASE}/ea6d_data/can"
OUTPUT_DIR = f"{BASE}/ea6d_output"

vae_ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/step1/model-*.pt'))
assert vae_ckpts, "No VAE checkpoints found — re-run Step 1 first."
vae_ckpt = vae_ckpts[-1]
print(f"Using VAE checkpoint: {vae_ckpt}")

diff_train_cfg = {
    'model': {
        'model_type': 'const_sde',
        'model_name': 'cond_unet',
        'image_size': [256, 256],
        'input_keys': ['image', 'cond'],
        'ckpt_path': None,
        'ignore_keys': [],
        'only_model': False,
        'timesteps': 1000,
        'train_sample': -1,
        'sampling_timesteps': 1,
        'loss_type': 'l2',
        'objective': 'pred_KC',
        'start_dist': 'normal',
        'perceptual_weight': 0,
        'scale_factor': 0.3,
        'scale_by_std': True,
        'default_scale': True,
        'scale_by_softsign': False,
        'eps': 1e-4,
        'weighting_loss': True,
        'use_disloss': True,
        'first_stage': {
            'embed_dim': 3,
            'lossconfig': {
                'disc_start': 50001,
                'kl_weight': 0.000001,
                'disc_weight': 0.5,
                'disc_in_channels': 3,
            },
            'ddconfig': {
                'double_z': True,
                'z_channels': 3,
                'resolution': [320, 320],
                'in_channels': 3,
                'out_ch': 3,
                'ch': 128,
                'ch_mult': [1, 2, 4],
                'num_res_blocks': 2,
                'attn_resolutions': [],
                'dropout': 0.0,
            },
            'ckpt_path': vae_ckpt,
        },
        'unet': {
            'dim': 128,
            'cond_net': 'swin',
            'fix_bb': False,
            'channels': 3,
            'out_mul': 1,
            'dim_mults': [1, 2, 4, 4],
            'cond_in_dim': 3,
            'cond_dim': 128,
            'cond_dim_mults': [2, 4],
            'window_sizes1': [[8, 8], [4, 4], [2, 2], [1, 1]],
            'window_sizes2': [[8, 8], [4, 4], [2, 2], [1, 1]],
            'fourier_scale': 16,
            'cond_pe': False,
            'num_pos_feats': 128,
            'cond_feature_size': [80, 80],
            'input_size': [80, 80],
        },
    },
    'data': {
        'name': 'edge',
        'crop_type': 'rand_resize_crop',
        'img_folder': DATA_DIR,
        'augment_horizontal_flip': True,
        'batch_size': 1,
        'num_workers': 2,
    },
    'trainer': {
        'gradient_accumulate_every': 8,
        'lr': 2e-4,
        'min_lr': 5e-6,
        'train_num_steps': 20000,
        'save_and_sample_every': 2000,
        'enable_resume': True,
        'log_freq': 500,
        'results_folder': f'{OUTPUT_DIR}/step2/',
        'amp': False,
        'fp16': False,
        'resume_milestone': 0,
        'test_before': False,
        'ema_update_after_step': 25000,
        'ema_update_every': 5000,
    },
}
with open('configs/colab_diff_train.yaml', 'w') as f:
    yaml.dump(diff_train_cfg, f, default_flow_style=False)
print("Diffusion config written.")

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
BASE = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(f'{BASE}/EA6D')
!python train_diffusionPart.py --cfg configs/colab_diff_train.yaml

## 5. Step 3 — Inference

Generate reconstructed images from test inputs using the trained diffusion model.

In [ ]:
import glob, yaml, os

BASE = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(f'{BASE}/EA6D')

OUTPUT_DIR = f"{BASE}/ea6d_output"
TEST_DIR = f"{BASE}/ea6d_data/can_test"

diff_ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/step2/model-*.pt'))
assert diff_ckpts, "No diffusion checkpoints found — re-run Step 2 first."
diff_ckpt = diff_ckpts[-1]
print(f"Using diffusion checkpoint: {diff_ckpt}")

with open('configs/colab_diff_train.yaml') as f:
    sample_cfg = yaml.safe_load(f)

sample_cfg['data'] = {
    'name': 'edge',
    'img_folder': TEST_DIR,
    'augment_horizontal_flip': False,
    'batch_size': 1,
    'num_workers': 2,
}
sample_cfg['sampler'] = {
    'sample_type': 'slide',
    'stride': [240, 240],
    'batch_size': 1,
    'sample_num': 50,
    'use_ema': False,
    'save_folder': f'{OUTPUT_DIR}/test_results/',
    'ckpt_path': diff_ckpt,
}
with open('configs/colab_sample.yaml', 'w') as f:
    yaml.dump(sample_cfg, f, default_flow_style=False)
print("Sample config written.")

In [ ]:
import os
BASE = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(f'{BASE}/EA6D')
!python sample_test.py --cfg configs/colab_sample.yaml

## 6. Results

In [ ]:
import glob, os
from IPython.display import display, Image as IPImage

BASE = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
OUTPUT_DIR = f"{BASE}/ea6d_output"
results = sorted(glob.glob(f'{OUTPUT_DIR}/**/*.png', recursive=True))
print(f"{len(results)} output images")
for img_path in results[:10]:
    print(img_path)
    display(IPImage(filename=img_path, width=512))